In [ ]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from collections.abc import Callable
from dataclasses import dataclass
from enum import IntEnum
from functools import lru_cache
from pathlib import Path

### Загрузка данных

In [ ]:
DATA = Path('../results_new.parquet')

# Читаем 10 колонок из 22: на 24 млн строк каждая лишняя float64 — это 194 МБ.
# xf*/yf* не используются нигде, а начальные условия x0..y2 нужны ровно в одной
# печатной таблице внизу — их дочитываем точечно (см. runs_ic).
COLS = ['delta1', 'delta2', 'eps', 'coupling_type', 'L', 'A', 'P', 's01', 's02', 's12']

# Наивный pd.read_parquet даёт пик 5.2 ГБ на фрейм в 1.9 ГБ. Три причины, все снимаются:
#   * jemalloc (пул arrow по умолчанию) не отдаёт освобождённые буферы ОС -> системный пул;
#   * потоки и pre_buffer держат в воздухе сразу несколько распакованных row group'ов;
#   * pandas копирует arrow-таблицу -> split_blocks + self_destruct отдают колонки по одной
#     и освобождают буфер сразу (после to_pandas таблица непригодна — это её цена).
# Итог: пик 3.0 ГБ, чтение на полсекунды дольше.
pa.set_memory_pool(pa.system_memory_pool())
_tbl = pq.read_table(DATA, columns=COLS, use_threads=False, pre_buffer=False)
df = _tbl.to_pandas(split_blocks=True, self_destruct=True)
del _tbl
df['coupling_type'] = df['coupling_type'].astype(np.int8)
print(df.shape)
df.head(3)

### Округление точности

In [ ]:
df['delta1'] = df['delta1'].round(6)
df['delta2'] = df['delta2'].round(6)
df['eps'] = df['eps'].round(6)

In [ ]:
COUPLING_NAMES = ['Инерционная', 'Инерционная (норм.)', 'Диссипативная', 'Диссипативная (норм.)']
# Подписи на графиках русские, а имена файлов — латиницей: кириллица в путях ломает
# \includegraphics в Overleaf.
COUPLING_SLUGS = ['inertial', 'inertialnorm', 'dissipative', 'dissipativenorm']
epsilons = sorted(df['eps'].unique())

class CouplingType(IntEnum):
    Inertial = 0
    InertialNorm = 1
    Dissipative = 2
    DissipativeNorm = 3

# Порог захвата. Сверху ограничен разрешением конечного окна: при |s| >= 2π/T_obs
# разность фаз за время наблюдения совершает хотя бы один полный проскок 2π —
# это несовместимо с определением захвата (ограниченность разности фаз).
# Снизу — остаточным наклоном в ядре языка (численный ноль, ≤ 2.5e-3).
# Обе границы задаются только длиной окна и не зависят ни от ε, ни от типа связи,
# поэтому θ здесь одна на всё (прежняя таблица 3×4 подобранных значений убрана).
T_OBS = 1600 - 240            # T - t_trans, см. simulation/experiments/multistability/config.yaml
THETA = 2 * np.pi / T_OBS     # ≈ 0.0046

# Максимум доли н.у. вне доминирующего режима. Мода — самое частое значение, а не
# большинство: при 30 н.у. и трёх равных кодах (10/10/10) мода = 1/3, вне моды = 2/3.
# Наблюдаемый максимум по данным — 0.667, поэтому шкала до 0.7, а не до 1.
DIS_MAX = 0.7

# Число запусков на точку печатается ниже, в ячейке агрегации: отдельная группировка
# 24-млн фрейма по тем же пяти ключам ради одной строчки диагностики не нужна.
print('epsilons:', epsilons)
print('coupling types:', sorted(df['coupling_type'].unique()))
print(f'T_obs = {T_OBS},  θ = 2π/T_obs = {THETA:.5f}')

In [ ]:
# --- Код режима ---------------------------------------------------------------
# Код строится по всем трём парам, но s12 НЕ является независимым измерением:
# из тождества (φ0-φ1) = (φ0-φ2) - (φ1-φ2) и линейности МНК-наклона следует
# s01 = s02 - s12 ТОЧНО (проверено: выполняется с машинной точностью на всех строках).
# Независимых величин две, а не три.
#
# Отсюда два следствия.
# 1) Комбинация «ровно две пары из трёх заперты» динамически невозможна: если Ω01=0
#    и Ω02=0, то Ω12=0 автоматически. Но порог заменяет «=0» на «<θ», а это свойство
#    при вычитании не сохраняется: |s12| <= |s01| + |s02| < 2θ. Третий наклон может
#    попасть в [θ, 2θ) и не пройти порог. Такие точки (>=2 бита из 3) сводим в один
#    код «полная».
# 2) Захват пары 1↔2 при незапертом хабе — это НЕ remote synchronization. Он держится
#    только на линии δ1=δ2, где листья идентичны и работает перестановочная симметрия
#    сети; ширина по расстройке там не превышает шага сетки. Вблизи центра полоса шире,
#    но лишь потому, что оба листа близки к захвату с хабом и обе частоты малы.
PAIRS = [('s01', '0–1'), ('s02', '0–2'), ('s12', '1–2')]
REGIME_LABELS = ['нет синхр.', '0↔1', '0↔2', '1↔2', 'полная (≥2 из 3 пар)']
REGIME_COLORS = ['#eeeeee', '#4477aa', '#66ccee', '#ccbb44', '#8b1a89']

_bits = np.column_stack([df[c].to_numpy() < THETA for c in ('s01', 's02', 's12')])
_raw = _bits @ np.array([1, 2, 4], dtype=np.int8)
df['code'] = np.select(
    [_bits.sum(1) >= 2, _raw == 1, _raw == 2, _raw == 4],
    [4, 1, 2, 3], default=0
).astype(np.int8)
del _bits, _raw

# --- Агрегация по точкам сетки -------------------------------------------------
# Один проход по (delta1, delta2, eps, тип связи) — дальше все карты слайсят PTS,
# а не сканируют df заново.
KEYS = ['delta1', 'delta2', 'eps', 'coupling_type']
_g = df.groupby(KEYS, sort=True)
PTS = _g.agg(
    L_max=('L', 'max'), L_min=('L', 'min'),
    A_max=('A', 'max'), A_min=('A', 'min'),
    P_max=('P', 'max'), P_min=('P', 'min'),
    s01=('s01', 'min'), s02=('s02', 'min'), s12=('s12', 'min'),
).reset_index()
for _v in ('L', 'A', 'P'):
    PTS[f'{_v}_spread'] = PTS[f'{_v}_max'] - PTS[f'{_v}_min']

# Номер точки сетки для каждой строки df: PTS.iloc[i] <-> GID == i. Дальше любая
# статистика по кодам — это bincount по GID, без повторных группировок 24-млн фрейма.
# int32 хватает (точек ~8e5) и вдвое дешевле int64; сам groupby с кэшем группировки
# после этого не нужен.
GID = _g.ngroup().to_numpy().astype(np.int32)
NPTS = len(PTS)
del _g


def code_counts(code, ncodes):
    """Длинная таблица счётчиков: по строке на каждый встретившийся (точка сетки, код).

    Порядок строк — по точке, внутри точки по коду, как у прежнего
    df.groupby(KEYS + ['code']).size(), но считается bincount'ом по GID: группировка
    24-млн фрейма по пяти ключам делалась заново на каждый порог θ.
    """
    cnt = np.bincount(GID * ncodes + code, minlength=NPTS * ncodes).reshape(NPTS, ncodes)
    gid, c = np.nonzero(cnt)
    return pd.DataFrame({'gid': gid, 'code': c, 'n': cnt[gid, c]})


def dominant(cnt):
    """Мода кода в каждой точке, выровненная по строкам PTS.

    Ничьи (например 15/15 по двум кодам) разрешаются порядком, который даёт
    sort_values (quicksort) — тот же вызов, что и раньше, поэтому карта не меняется.
    """
    dom = cnt.sort_values('n').drop_duplicates('gid', keep='last')
    return dom.set_index('gid')['code'].reindex(np.arange(NPTS)).to_numpy()


_cnt = code_counts(df['code'].to_numpy(), len(REGIME_LABELS))
_by_pt = _cnt.groupby('gid')['n']
PTS['dominant_code'] = dominant(_cnt)
PTS['multi'] = (_by_pt.size() > 1).to_numpy()
PTS['outside'] = 1 - (_by_pt.max() / _by_pt.sum()).to_numpy()

# Разбивка по кодам для ховера, по убыванию частоты.
_cnt['lbl'] = np.array(REGIME_LABELS)[_cnt['code']] + ': ' + _cnt['n'].astype(str)
# Различных строк всего несколько тысяч на ~8e5 точек — category вместо object экономит
# около сотни мегабайт.
PTS['breakdown'] = pd.Categorical(
    _cnt.sort_values('n', ascending=False).groupby('gid')['lbl'].agg('<br>'.join))

print('runs per point:', np.unique(_by_pt.sum().to_numpy()))
del _cnt, _by_pt


@lru_cache(maxsize=None)
def grid(col, eps, ct):
    """Столбец PTS -> (z, x, y) для heatmap в точке (eps, тип связи).

    Оси отдаём как ndarray, а не list: plotly 6 кладёт numpy-массивы в JSON
    base64-блоком, минуя float->str, — это заметно дешевле на 100+ фигурах экспорта.
    """
    d = PTS[(PTS['eps'] == eps) & (PTS['coupling_type'] == ct)]
    p = d.pivot(index='delta1', columns='delta2', values=col)
    return p.to_numpy(), p.columns.to_numpy(), p.index.to_numpy()


def cbar(title, x, **kw):
    return dict(title=title, x=x, len=kw.pop('len', 0.9), thickness=12, **kw)


def heat(col, eps, ct, hover='%{z:.3f}', **kw):
    z, x, y = grid(col, eps, ct)
    return go.Heatmap(z=z, x=x, y=y,
                      hovertemplate=f'δ₁=%{{y:.3f}} δ₂=%{{x:.3f}}<br>{hover}<extra></extra>', **kw)


def discrete_colorscale(colors):
    n = len(colors)
    return [pt for i, c in enumerate(colors) for pt in ([i / n, c], [(i + 1) / n, c])]


REGIME_SCALE = discrete_colorscale(REGIME_COLORS)


def lock_square(fig, ncols, nrows=1):
    # В subplot'ах у каждой панели своя пара осей (x/y, x2/y2, ...), поэтому общий
    # update_yaxes(scaleanchor='x') привязал бы все Y к первой X. Якорим попанельно:
    # δ₁ и δ₂ пробегают один диапазон, карта обязана быть квадратной.
    for r in range(1, nrows + 1):
        for c in range(1, ncols + 1):
            k = (r - 1) * ncols + c
            fig.update_yaxes(scaleanchor=('x' if k == 1 else f'x{k}'), scaleratio=1,
                             constrain='domain', row=r, col=c)
    return fig


# --- Сборка фигур --------------------------------------------------------------
# Каждая карта нужна в двух видах: интерактивном (слайдеры по ε и типу связи) и
# статическом (по файлу на ε/тип связи для статьи). Различаются они только тем,
# какие трейсы видимы, поэтому фигура описывается один раз — FigSpec, — а
# animated_fig / static_fig / alleps_fig собирают из неё нужный вид.
@dataclass(frozen=True)
class FigSpec:
    skeleton: Callable[[], go.Figure]        # пустая сетка subplot'ов
    traces: Callable[[float], list]          # ε -> трейсы всех типов связи подряд
    place: Callable[[int], tuple]            # номер трейса в группе -> (row, col)
    layout: Callable[..., go.Figure]         # оси, размеры, заголовок
    group: int = 0                           # трейсов на один тип связи (0 — тип связи не измерение фигуры)
    legend: Callable[[go.Figure], int] = None  # доп. трейсы-легенда, возвращает их число
    slider_margin_b: int = 0                 # место под слайдеры в интерактивном виде


def static_fig(spec, eps, ct=None, title=None):
    fig = spec.skeleton()
    trs = spec.traces(eps)
    if ct is not None:
        trs = trs[ct * spec.group:(ct + 1) * spec.group]
    for i, tr in enumerate(trs):
        fig.add_trace(tr, row=spec.place(i)[0], col=spec.place(i)[1])
    if spec.legend:
        spec.legend(fig)
    return spec.layout(fig, title)


def animated_fig(spec, title=None):
    fig = spec.skeleton()
    for i, tr in enumerate(spec.traces(epsilons[0])):
        if spec.group:
            tr.visible = i // spec.group == 0
        fig.add_trace(tr, row=spec.place(i)[0], col=spec.place(i)[1])
    n_extra = spec.legend(fig) if spec.legend else 0
    spec.layout(fig, title)
    if spec.slider_margin_b:
        fig.update_layout(margin_b=spec.slider_margin_b)

    frames = [go.Frame(name=str(e), data=spec.traces(e)) for e in epsilons]
    steps = [dict(method='animate', label=f.name,
                  args=[[f.name], dict(mode='immediate', frame=dict(duration=0, redraw=True),
                                       transition=dict(duration=0))])
             for f in frames]
    fig.frames = frames
    sliders = [dict(active=0, pad={'t': 60}, currentvalue={'prefix': 'ε = '}, steps=steps)]
    if spec.group:
        total = len(COUPLING_NAMES) * spec.group
        sliders = [dict(s, yanchor='top', y=0, pad={'t': 40}) for s in sliders] + [dict(
            active=0, yanchor='top', y=0, pad={'t': 110}, currentvalue={'prefix': 'тип связи: '},
            steps=[dict(method='restyle', label=name,
                        args=[{'visible': [(ci * spec.group <= i < (ci + 1) * spec.group)
                                           if i < total else True
                                           for i in range(total + n_extra)]}])
                   for ci, name in enumerate(COUPLING_NAMES)])]
    return fig.update_layout(sliders=sliders)


def alleps_fig(spec, ct, colorbars, **layout):
    """Сводная фигура: колонки — ε, строки — панели одного типа связи.

    Так в статье не приходится произвольно выбирать пару значений ε для иллюстрации.
    Оси общие: подписи и деления только по краям сетки, иначе при крупном шрифте
    (fit_fonts масштабирует его под ширину канвы) они наползают друг на друга.
    """
    nrows, last = len(colorbars), len(epsilons) - 1
    fig = make_subplots(rows=nrows, cols=len(epsilons),
                        subplot_titles=[f'ε = {e:.2f}' for e in epsilons] + [''] * len(epsilons),
                        shared_xaxes=True, shared_yaxes=True,
                        horizontal_spacing=0.02, vertical_spacing=0.07)
    for j, eps in enumerate(epsilons):
        for r, (tr, cb) in enumerate(zip(spec.traces(eps)[ct * spec.group:(ct + 1) * spec.group],
                                         colorbars), start=1):
            tr.showscale = cb is not None and j == last
            if tr.showscale:
                tr.colorbar = cb
            fig.add_trace(tr, row=r, col=j + 1)
    if spec.legend:
        spec.legend(fig)
    lock_square(fig, len(epsilons), nrows=nrows)
    fig.update_xaxes(title_text='δ₂', row=nrows)
    fig.update_yaxes(title_text='δ₁', col=1)
    return fig.update_layout(height=700, width=1700, autosize=False, **layout)


# --- Размер шрифта в экспортируемых фигурах -----------------------------------
# Kaleido пишет PDF в пунктах: 1 px канвы = 1 pt. В статье фигура вставляется как
# \includegraphics[width=\textwidth], то есть ужимается с width до TEXTWIDTH_PT,
# и вместе с ней ужимается шрифт. Чтобы в печати получить TARGET_PT, задаём на
# канве шрифт, увеличенный в (width / TEXTWIDTH_PT) раз. Для сеток из многих
# панелей 10 pt получается слишком плотно, поэтому держим подписи немного мельче.
TEXTWIDTH_PT = 484          # \textwidth статьи: A4 210мм минус поля 25+15 => 170мм
TARGET_PT = 8               # желаемый кегль подписей в готовом PDF


def fit_fonts(fig, width):
    base = TARGET_PT * width / TEXTWIDTH_PT
    fig.update_layout(font=dict(size=base))
    # Заголовки панелей и подписи рядов — это annotations, им нужен свой размер.
    for ann in (fig.layout.annotations or []):
        ann.font = dict(size=base * 0.95)
    for ax in fig.layout:
        if ax.startswith(('xaxis', 'yaxis')):
            fig.layout[ax].title.font = dict(size=base)
            fig.layout[ax].tickfont = dict(size=base * 0.85)
    for tr in fig.data:
        cb = getattr(tr, 'colorbar', None)
        if cb is not None:
            cb.title.font = dict(size=base)
            cb.tickfont = dict(size=base * 0.85)
    return fig


# Длинные названия связей не влезают в узкую колонку — переносим по строкам.
COUPLING_NAMES_BR = [n.replace(' (', '<br>(') for n in COUPLING_NAMES]

## Подбор порога θ

### Распределение наклонов разности фаз

In [ ]:
# Распределение |s| по реальным рёбрам звезды (s01, s02). s12 не включаем:
# он не независим (s01 = s02 - s12 тождественно), иначе в гистограмму подмешивается
# производная величина.
# Обе оси логарифмические: запертая популяция живёт в 1e-6..1e-3, и на линейной
# шкале [0, 0.2] она целиком укладывается в первые полпроцента оси — провал не виден.
_BINS = np.logspace(-6, 0, 73)
_CTR = np.sqrt(_BINS[:-1] * _BINS[1:])
_DLOG = np.diff(np.log10(_BINS))


@lru_cache(maxsize=None)
def slope_hist(eps, ct):
    # Кэш: те же 20 гистограмм строятся заново для кадров слайдера и для экспорта,
    # а каждая — это проход по 24-млн колонке.
    g = df[(df['eps'] == eps) & (df['coupling_type'] == ct)]
    s = np.concatenate([g['s01'].to_numpy(), g['s02'].to_numpy()])
    h, _ = np.histogram(s[s > 0], bins=_BINS)
    return h / h.sum() / _DLOG          # плотность на порядок величины (на единицу log10|s|)


def slope_hist_layout(fig, title):
    fig.update_xaxes(type='log', title_text='|s|', range=[-6, 0])
    fig.update_yaxes(type='log', range=[-2.5, 0.7])
    fig.update_yaxes(title_text='плотность на порядок величины', col=1)
    for c in range(1, 5):
        fig.add_vline(x=THETA, line=dict(color='green', dash='dash', width=2), row=1, col=c)
    return fig.update_layout(height=400, width=1300, showlegend=False, title=title,
                             margin_t=100 if title else 75)


SLOPE_HIST = FigSpec(
    skeleton=lambda: make_subplots(rows=1, cols=4, subplot_titles=COUPLING_NAMES_BR,
                                   horizontal_spacing=0.03, shared_yaxes=True),
    traces=lambda eps: [go.Scatter(x=_CTR, y=slope_hist(eps, ct), mode='lines',
                                   line=dict(color='steelblue', shape='hv'), fill='tozeroy')
                        for ct in range(4)],
    place=lambda i: (1, i + 1),
    layout=slope_hist_layout,
)

animated_fig(SLOPE_HIST,
             f'Распределение наклонов s01, s02 (зелёная линия — θ = 2π/T_obs = {THETA:.4f})').show()

In [ ]:
print(f'θ = 2π/T_obs = {THETA:.5f}   (единая для всех ε и всех типов связи)')

### Устойчивость карты к выбору θ

θ задана физически: $\theta = 2\pi/T_{obs}$ — порог, выше которого «захват» допускал бы
полный проскок фазы на $2\pi$ за окно наблюдения. Автоподбор по плато убран.

Проверка: доля области захвата, меняющая режим при **четырёхкратном** сдвиге порога
($\theta/2 \to 2\theta$). Знаменатель — ячейки, запертые хоть при одном из порогов
(фон, который никогда не заперт, разбавляет метрику и исключён).

In [ ]:
def dominant_at(th):
    """Мода 2-битного кода (по s01, s02) в каждой точке сетки при пороге th.

    Считаем bincount'ом по готовым GID: df.assign(code=...).groupby(...) копировал
    весь 24-млн фрейм на каждый порог (несколько ГБ, пик RSS втрое выше рабочего).
    """
    code = ((df['s01'].to_numpy() < th).astype(np.int8)
            | ((df['s02'].to_numpy() < th).astype(np.int8) << 1))
    return dominant(code_counts(code, 4))


_lo, _hi = dominant_at(THETA / 2), dominant_at(THETA * 2)
_active = (_lo > 0) | (_hi > 0)               # заперта хоть при одном из порогов
_r = (pd.DataFrame({'flip': (_lo != _hi) & _active, 'active': _active,
                    'eps': PTS['eps'], 'coupling_type': PTS['coupling_type']})
      .groupby(['eps', 'coupling_type']).sum())

THETA_ROBUST = (_r['flip'] / _r['active']).unstack('coupling_type')
THETA_ROBUST.columns = COUPLING_NAMES
THETA_ROBUST.style.format('{:.1%}').background_gradient(cmap='Reds', vmin=0, vmax=0.7)

### Карта наклона и устойчивость к θ

`min|slope|` по н.у. (плато ≈ 0 — захват) + контуры θ ∈ {0.01…0.05}. Контуры сбиты в тонкую полосу ⇒ граница захвата не зависит от выбора θ.

In [ ]:
def slope_map_traces(eps):
    traces = []
    for ct in range(4):
        for pi, (col, _) in enumerate(PAIRS):
            z, x, y = grid(col, eps, ct)
            traces.append(heat(col, eps, ct, hover='%{z:.4f}',
                               colorscale='Viridis', zmin=0, zmax=0.06, showscale=(pi == 0),
                               colorbar=cbar('мин. |s|', 1.02)))
            traces.append(go.Contour(
                z=z, x=x, y=y,
                contours=dict(start=0.01, end=0.05, size=0.01, coloring='lines'),
                line=dict(width=1, color='white'), showscale=False, hoverinfo='skip'))
    return traces


def slope_map_layout(fig, title):
    fig.update_xaxes(title_text='δ₂')
    fig.update_yaxes(title_text='δ₁', col=1)
    lock_square(fig, 3)
    return fig.update_layout(height=450, width=1300, title=title, margin_t=100 if title else 40)


SLOPE_MAP = FigSpec(
    skeleton=lambda: make_subplots(rows=1, cols=3, subplot_titles=[lbl for _, lbl in PAIRS],
                                   horizontal_spacing=0.06),
    traces=slope_map_traces,
    place=lambda i: (1, (i % 6) // 2 + 1),
    layout=slope_map_layout,
    group=6,
    slider_margin_b=140,
)

animated_fig(SLOPE_MAP, 'Минимальный по н.у. наклон |s| и контуры θ').show()

## Карты режимов синхронизации

In [ ]:
_LBL = np.array(REGIME_LABELS, dtype=object)


def regime_legend(fig):
    # Дискретная heatmap не создаёт нормальную легенду категорий сама.
    for lbl, col in zip(REGIME_LABELS, REGIME_COLORS):
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode='markers', name=lbl,
            marker=dict(size=12, symbol='square', color=col,
                        line=dict(width=0.5, color='#888')),
            showlegend=True, hoverinfo='skip'), row=1, col=1)
    return len(REGIME_LABELS)


def regime_traces(eps):
    traces = []
    for ct in range(4):
        dom = grid('dominant_code', eps, ct)[0]
        custom = np.dstack([_LBL[dom.astype(int)],
                            grid('breakdown', eps, ct)[0],
                            grid('outside', eps, ct)[0]])
        traces.append(heat('dominant_code', eps, ct,
                           hover='мода: %{customdata[0]}<br>вне моды: %{customdata[2]:.3f}'
                                 '<br>%{customdata[1]}',
                           customdata=custom, colorscale=REGIME_SCALE,
                           zmin=-0.5, zmax=4.5, showscale=False))
        traces.append(heat('outside', eps, ct, hover='вне моды: %{z:.3f}',
                           colorscale='Viridis', zmin=0, zmax=DIS_MAX, showscale=True,
                           colorbar=cbar('вне моды', 1.01, len=0.78)))
    return traces


def regime_layout(fig, title):
    fig.update_xaxes(title_text='δ₂')
    fig.update_yaxes(title_text='δ₁', col=1)
    lock_square(fig, 2)
    return fig.update_layout(
        height=620, width=1350, autosize=False, title=title,
        plot_bgcolor='white', margin=dict(t=95, b=150, r=260), showlegend=True,
        legend=dict(orientation='v', x=1.10, y=0.98, xanchor='left', yanchor='top',
                    bgcolor='rgba(255,255,255,0)', title_text='режим'))


REGIME = FigSpec(
    skeleton=lambda: make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
                                   subplot_titles=['мода режима', 'доля н.у. вне моды']),
    traces=regime_traces,
    place=lambda i: (1, i % 2 + 1),
    layout=regime_layout,
    group=2,
    legend=regime_legend,
)

animated_fig(REGIME, 'Режимы синхронизации: мода режима и доля н.у. вне моды').show()

## Целевые функции

In [ ]:
A_EDGES = np.linspace(df['A'].min(), df['A'].max(), 61)
P_EDGES = np.linspace(df['P'].min(), df['P'].max(), 61)
_AC = 0.5 * (A_EDGES[:-1] + A_EDGES[1:])
_PC = 0.5 * (P_EDGES[:-1] + P_EDGES[1:])

_PA_HIST = {}
for (_e, _ct), _g in df.groupby(['eps', 'coupling_type']):
    _h, _, _ = np.histogram2d(_g['A'].to_numpy(), _g['P'].to_numpy(), bins=[A_EDGES, P_EDGES])
    _PA_HIST[(_e, _ct)] = np.log1p(_h.T)


def pa_density_layout(fig, title):
    fig.update_xaxes(title_text='A')
    fig.update_yaxes(title_text='P', col=1)
    return fig.update_layout(height=400, width=1300, title=title, margin_t=100 if title else 75)


PA_DENSITY = FigSpec(
    skeleton=lambda: make_subplots(rows=1, cols=4, subplot_titles=COUPLING_NAMES_BR,
                                   horizontal_spacing=0.04, shared_yaxes=True),
    traces=lambda eps: [go.Heatmap(
        z=_PA_HIST[(eps, ct)], x=_AC, y=_PC, colorscale='Blues', showscale=False,
        hovertemplate='A=%{x:.3f} P=%{y:.3f}<br>log(1+n)=%{z:.2f}<extra></extra>')
        for ct in range(4)],
    place=lambda i: (1, i + 1),
    layout=pa_density_layout,
)

animated_fig(PA_DENSITY, 'Совместное распределение P и A (логарифм плотности запусков)').show()

In [ ]:
def pa_corr(label, mask=None):
    corr = ((df if mask is None else df[mask])
            .groupby(['eps', 'coupling_type'])[['A', 'P']]
            .apply(lambda g: pd.Series({'ρ_Pearson': g['A'].corr(g['P']),
                                        'ρ_Spearman': g['A'].corr(g['P'], method='spearman')}))
            .reset_index())
    corr['coupling'] = corr['coupling_type'].map(dict(enumerate(COUPLING_NAMES)))
    corr['subset'] = label
    return corr


PA_CORR = pd.concat([pa_corr('в захвате', df['code'] != 0), pa_corr('вся карта')],
                    ignore_index=True)

# Матрицы ε × тип связи для двух подвыборок; шкала общая, поэтому zmin берём по обеим.
PA_SPEARMAN = [(sub, g.pivot(index='eps', columns='coupling', values='ρ_Spearman')[COUPLING_NAMES])
               for sub, g in PA_CORR.groupby('subset', sort=False)]
PA_ZMIN = float(min(m.to_numpy().min() for _, m in PA_SPEARMAN))


def fig_pa_corr(title=None):
    fig = make_subplots(rows=1, cols=2, subplot_titles=[sub for sub, _ in PA_SPEARMAN],
                        horizontal_spacing=0.10, shared_yaxes=True)
    for col, (_, m) in enumerate(PA_SPEARMAN, start=1):
        fig.add_trace(go.Heatmap(
            z=m.to_numpy(), x=m.columns.to_numpy(), y=[f'{e:.2f}' for e in m.index],
            text=m.to_numpy(), texttemplate='%{text:.3f}',
            colorscale='RdYlGn', zmin=PA_ZMIN, zmax=1.0,
            colorbar=dict(title='ρ Спирмена', x=1.02), showscale=(col == 2)), row=1, col=col)
    fig.update_xaxes(title_text='тип связи')
    fig.update_yaxes(title_text='ε', col=1)
    return fig.update_layout(height=430, width=1250, title=title,
                             margin=dict(t=80 if title else 70, b=90, l=70, r=120))


fig_pa_corr('Корреляция Спирмана A и P по ε и типу связи').show()

## Синфазность

Две величины по начальным условиям в каждой точке сетки:

- **max $P$** — достигается ли синфазный режим хоть при каких-то н.у.;
- **$\Delta P$ = max − min** — насколько синфазность зависит от выбора н.у.

Медиана и квантили здесь неинформативны: если синфазный режим существует, но узок по
бассейну притяжения, они его просто не видят.

In [ ]:
P_LO = float(PTS['P_min'].min())
P_SPREAD_HI = float(PTS['P_spread'].max())


def inphase_traces(eps):
    traces = []
    for ct in range(4):
        # Pmax — есть ли синфазный режим хоть при каких-то н.у. в этой точке;
        # Pmax - Pmin — насколько синфазность зависит от начальных условий.
        traces.append(heat('P_max', eps, ct, colorscale='Viridis', zmin=P_LO, zmax=1.0,
                           showscale=True, colorbar=cbar('макс. P', 0.44)))
        traces.append(heat('P_spread', eps, ct, colorscale='Viridis', zmin=0.0, zmax=P_SPREAD_HI,
                           showscale=True, colorbar=cbar('разброс P', 1.0)))
    return traces


def inphase_layout(fig, title):
    fig.update_xaxes(title_text='δ₂')
    fig.update_yaxes(title_text='δ₁', col=1)
    lock_square(fig, 2)
    return fig.update_layout(height=600, width=1200, title=title, margin_t=100 if title else 40)


INPHASE = FigSpec(
    skeleton=lambda: make_subplots(
        rows=1, cols=2, horizontal_spacing=0.18,
        subplot_titles=['максимум P по н.у.', 'разброс P: максимум − минимум']),
    traces=inphase_traces,
    place=lambda i: (1, i % 2 + 1),
    layout=inphase_layout,
    group=2,
    slider_margin_b=140,
)

animated_fig(INPHASE, 'Синфазность в точке: максимум и разброс P по 30 начальным условиям').show()

## Исследование $\Delta L$, $\Delta A$, $\Delta P$ как критериев мультистабильности

In [ ]:
SPREADS = [('L_spread', 'ΔL'), ('A_spread', 'ΔA'), ('P_spread', 'ΔP')]
_REG = [(False, 'один режим', 'steelblue'), (True, 'мультистаб.', 'tomato')]


def spread_hist_traces(eps):
    a = PTS[PTS['eps'] == eps]
    return [go.Histogram(
        x=a.loc[(a['coupling_type'] == ct) & (a['multi'] == is_multi), col], nbinsx=50,
        marker_color=color, opacity=0.6, name=name, legendgroup=name,
        showlegend=(si == 0 and ct == 0))
        for si, (col, _) in enumerate(SPREADS)
        for ct in range(4)
        for is_multi, name, color in _REG]


def spread_hist_layout(fig, title):
    fig.update_yaxes(type='log', nticks=4, title_text='')
    fig.update_xaxes(title_text='', nticks=4)
    fig.add_annotation(text='разброс по 30 н.у.', x=0.5, y=-0.08, xref='paper', yref='paper',
                       showarrow=False, xanchor='center', yanchor='top')
    fig.add_annotation(text='число точек сетки', x=-0.06, y=0.5, xref='paper', yref='paper',
                       showarrow=False, textangle=-90, xanchor='center', yanchor='middle')
    return fig.update_layout(height=660, width=1300, barmode='overlay', title=title,
                             margin=dict(l=130, r=150, t=80 if title else 75, b=125))


SPREAD_HIST = FigSpec(
    skeleton=lambda: make_subplots(rows=3, cols=4, row_titles=[lbl for _, lbl in SPREADS],
                                   column_titles=COUPLING_NAMES_BR,
                                   horizontal_spacing=0.05, vertical_spacing=0.07),
    traces=spread_hist_traces,
    place=lambda i: (i // 8 + 1, (i % 8) // 2 + 1),
    layout=spread_hist_layout,
)

animated_fig(SPREAD_HIST, 'Разброс L, A, P в одно- и мультистабильных точках').show()

## Автоматический экспорт фигур без слайдеров

Следующая ячейка экспортирует статические фигуры в PDF (для Overleaf: `\includegraphics`
понимает их без дополнительных пакетов) и в SVG (для просмотра). В каждом файле
зафиксированы конкретные $ε$ и/или тип связи, поэтому интерактивные слайдеры не нужны.

Экспорт требует пакет `kaleido`. Если ячейка падает с ошибкой про Kaleido, установи его
в окружение `display` (`uv add kaleido`) и перезапусти экспорт.

In [ ]:
EXPORT_DIR = Path('../plots/export')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Окно зума в центр. Верхняя граница шире нижней: у ненормированной связи центр
# языка захвата сидит не в нуле, а в δ ≈ ε/2 (сдвиг от степени узла).
ZOOM = (-0.10, 0.15)

DEMO_RUNS = [
    ('δ₁=-0.20 δ₂=0.12 — код 2 (0↔2), L≈0.834', '../simulation/demo/small_l_code2.csv'),
    ('δ₁=-0.20 δ₂=0.12 — код 0 (нет синхр.), L≈0.838', '../simulation/demo/small_l_code0.csv'),
    ('δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈2.1', '../simulation/demo/large_dl_high_l.csv'),
    ('δ₁=-0.50 δ₂=-0.50 — код 4 (1↔2), L≈0.235', '../simulation/demo/large_dl_low_l.csv'),
]
DEMO_WINDOW = (600.0, 640.0)          # установившийся режим: [t_trans, t_trans + 40]


def fig_timeseries(title=None):
    colors = {'x0': '#e41a1c', 'x1': '#4daf4a', 'x2': '#377eb8'}
    lo, hi = DEMO_WINDOW
    fig = make_subplots(rows=len(DEMO_RUNS), cols=1, shared_xaxes=True,
                        subplot_titles=[t for t, _ in DEMO_RUNS])
    for ri, (_, path) in enumerate(DEMO_RUNS):
        d = pd.read_csv(path, comment='#')
        steady = d[(d['t'] >= lo) & (d['t'] < hi)]
        for col, c in colors.items():
            fig.add_trace(go.Scatter(x=steady['t'], y=steady[col], mode='lines',
                                     line=dict(color=c, width=1), name=col,
                                     legendgroup=col, showlegend=(ri == 0)),
                          row=ri + 1, col=1)
    fig.update_xaxes(title_text='t', row=len(DEMO_RUNS), col=1)
    fig.update_yaxes(title_text='x')
    return fig.update_layout(height=1000, width=1300, title=title,
                             margin_t=100 if title else 40)


def _zoomed(fig):
    fig = go.Figure(fig)
    fig.update_xaxes(range=list(ZOOM))
    fig.update_yaxes(range=list(ZOOM))
    # У фигур без заголовка (режимы, синфазность) не воскрешаем его: то, что это
    # центральная область, сказано в \caption.
    if fig.layout.title.text:
        fig.update_layout(title=fig.layout.title.text + ' — центральная область')
    return fig


def static_figures():
    """{имя файла: фигура} — всё, что уходит в статью. Заголовки не рисуем: в статье
    их роль играет \\caption, дублировать нельзя."""
    figs = {}
    for eps in epsilons:
        et = f"eps-{eps:.2f}".replace('.', 'p')
        figs[f'slope-hist-{et}'] = static_fig(SLOPE_HIST, eps)
        figs[f'pa-density-{et}'] = static_fig(PA_DENSITY, eps)
        figs[f'spread-hist-{et}'] = static_fig(SPREAD_HIST, eps)
        for ct, slug in enumerate(COUPLING_SLUGS):
            tag = f'{et}-{slug}'
            figs[f'slope-map-{tag}'] = static_fig(SLOPE_MAP, eps, ct)
            for spec, name in ((REGIME, 'regime'), (INPHASE, 'inphase')):
                fig = static_fig(spec, eps, ct)
                figs[f'{name}-{tag}'] = fig
                figs[f'{name}-{tag}-zoom'] = _zoomed(fig)

    # Сводные: все ε в одной фигуре, по одной на тип связи.
    for ct, slug in enumerate(COUPLING_SLUGS):
        reg = alleps_fig(
            REGIME, ct, [None, cbar('вне моды', 1.02, y=0.25, len=0.45)],
            plot_bgcolor='white', margin=dict(t=60, b=150, r=170), showlegend=True,
            legend=dict(orientation='h', x=0.5, y=-0.22, xanchor='center',
                        bgcolor='rgba(255,255,255,0)', title_text='режим'))
        figs[f'regime-alleps-{slug}'] = reg
        figs[f'regime-alleps-{slug}-zoom'] = _zoomed(reg)
        figs[f'inphase-alleps-{slug}'] = alleps_fig(
            INPHASE, ct, [cbar('макс. P', 1.02, y=0.78, len=0.42),
                          cbar('разброс P', 1.02, y=0.24, len=0.42)],
            margin=dict(t=60, b=70, r=190))

    figs['pa-corr'] = fig_pa_corr()
    figs['timeseries'] = fig_timeseries()
    return {name: fit_fonts(f, f.layout.width or TEXTWIDTH_PT) for name, f in figs.items()}


def export_all_static():
    # PDF — для Overleaf: pdflatex вставляет его \includegraphics'ом без доп. пакетов.
    # SVG — для просмотра в браузере. В обоих heatmap остаётся растром (kaleido
    # встраивает её как <image>), векторными выходят оси, подписи и легенда.
    #
    # Пишем ОДНИМ вызовом write_images: fig.write_image поднимает и гасит браузер kaleido
    # на каждый файл, а это 258 запусков и ~9 минут против ~1 минуты на пакетную запись,
    # где браузер переиспользуется. Размеры канвы передаём явно: в отличие от write_image,
    # пакетный вызов НЕ берёт их из layout и молча рисует всё дефолтными 700x500.
    figs = static_figures()
    names = list(figs)
    pio.write_images(
        [figs[n] for n in names] * 2,
        [EXPORT_DIR / f'{n}.pdf' for n in names] + [EXPORT_DIR / f'{n}.svg' for n in names],
        format=['pdf'] * len(names) + ['svg'] * len(names),
        width=[figs[n].layout.width for n in names] * 2,
        height=[figs[n].layout.height for n in names] * 2)

    # kaleido начинает SVG сразу с <svg>, без XML-декларации. Байты — UTF-8, но без
    # объявления кодировки просмотрщик угадывает cp1252, и δ/− превращаются в кракозябры.
    for name in names:
        svg = EXPORT_DIR / f'{name}.svg'
        svg.write_bytes(b'<?xml version="1.0" encoding="utf-8"?>\n' + svg.read_bytes())
    print(f'Exported {len(names)} figures (PDF + SVG) to {EXPORT_DIR}')
    return names


# При Run All экспорт выполняется автоматически. Если нужно только открыть ноутбук
# без долгой записи файлов, поставь RUN_STATIC_EXPORT = False.
RUN_STATIC_EXPORT = True
if RUN_STATIC_EXPORT:
    EXPORTED_STATIC = export_all_static()

In [ ]:
IC_COLS = ['x0', 'y0', 'x1', 'y1', 'x2', 'y2']


def runs_at(eps, ct, d1, d2):
    return df[(df['eps'] == eps) & (df['coupling_type'] == ct)
              & np.isclose(df['delta1'], d1) & np.isclose(df['delta2'], d2)]


def runs_ic(eps, ct, d1, d2):
    """Начальные условия для одной точки сетки.

    В df их не держим: 6 колонок float64 на 24 млн строк — это 1.2 ГБ ради одной
    печатной таблицы. Дочитываем из parquet один тип связи (по нему предикат
    проталкивается в файл; eps там лежит как 0.0299999999999999, фильтровать по нему
    нельзя — отбираем уже в pandas, после округления).
    """
    d = pd.read_parquet(DATA, columns=KEYS + IC_COLS, filters=[('coupling_type', '==', ct)])
    m = ((d['eps'].round(6) == eps)
         & np.isclose(d['delta1'], d1) & np.isclose(d['delta2'], d2))
    return d.loc[m, IC_COLS].reset_index(drop=True)


def counterexamples(eps, ct=0):
    d = PTS[(PTS['eps'] == eps) & (PTS['coupling_type'] == ct)]
    return (d[d['multi'] & (d['L_spread'] < 0.1)].head(3),
            d[~d['multi'] & (d['L_spread'] > 0.5)].head(3))


def show_counterexamples(eps, ct=0):
    ms, ml = counterexamples(eps, ct)
    print(f'=== ε={eps}, {COUPLING_NAMES[ct]} ===')
    print('Мультистаб. с малым ΔL (<0.1):')
    for _, r in ms.iterrows():
        codes = sorted(runs_at(eps, ct, r.delta1, r.delta2)['code'].unique())
        print(f'  δ1={r.delta1:.2f} δ2={r.delta2:.2f} ΔL={r.L_spread:.4f} коды={codes}')
    print('Один режим с большим ΔL (>0.5):')
    for _, r in ml.iterrows():
        L = runs_at(eps, ct, r.delta1, r.delta2)['L'].round(3).tolist()
        print(f'  δ1={r.delta1:.2f} δ2={r.delta2:.2f} ΔL={r.L_spread:.4f} L={L}')


show_counterexamples(epsilons[2])

### Проверка точки с маленьким $\Delta L$, но мультистабильностью

In [ ]:
def show_runs_at(eps, kind, ct=0):
    ms, ml = counterexamples(eps, ct)
    sel = ms if kind == 'multi_small' else ml
    if len(sel) == 0:
        print('нет таких точек в данных'); return
    r0 = sel.iloc[0]
    pt = runs_at(eps, ct, r0.delta1, r0.delta2)[['code', 'L']].reset_index(drop=True)
    ic = runs_ic(eps, ct, r0.delta1, r0.delta2)     # н.у. читаем из parquet точечно
    print(f'ε={eps} δ1={r0.delta1:.2f} δ2={r0.delta2:.2f} ({COUPLING_NAMES[ct]}), ΔL={r0.L_spread:.4f}')
    print(pd.concat([pt, ic], axis=1).round(3).to_string(index=False))


show_runs_at(epsilons[2], 'multi_small')

### Проверка точки с высоким $\Delta L$, но однорежимностью

In [ ]:
show_runs_at(epsilons[2], 'mono_large')

In [ ]:
fig_timeseries('Осцилляции в установившемся режиме').show()